# Parameter Estimation — Joint & Dual Filters

*Course 3 — Nonlinear Kalman Filters, Part 3. So far we estimated the **state** assuming the model was known. Often the model's **parameters** are unknown too (physical constants, aging coefficients). The same nonlinear filters ([EKF](12_Extended_Kalman_Filter.ipynb) / [SPKF](13_Sigma_Point_Unscented_Kalman_Filter.ipynb)) can estimate parameters — or state **and** parameters together.*

**Style:** every equation gets a plain-language paraphrase (→); extra intuition is flagged **→ Intuition**.

### 🧩 A State-Space Model for Parameters

- Treat the unknown parameter vector $\theta$ as the "state" of a trivial dynamic model:

$$
\theta_{k+1} = \theta_k + r_k, \qquad d_k = g(x_k, u_k, \theta_k) + e_k,
$$

  where $r_k$ is (artificial) process noise and $e_k$ is measurement noise.

  → The **process equation is a random walk**: parameters are "constant, but allowed to drift a little." The **output equation** $g$ relates the parameters to a measurable quantity $d_k$ (usually via the system's own $f,h$).

- **→ Intuition:** by casting parameters as a slowly-moving state, the *entire Kalman toolbox* applies — the only new modeling choice is how much drift $r_k$ to allow.

### 🧩 The Role of the Artificial Noise $r_k$

- Even if the true parameters are constant, we deliberately keep $\Sigma_{\tilde{r}} = \operatorname{cov}(r_k) > 0$.

  → Nonzero $r_k$ keeps the parameter covariance $\Sigma_{\tilde\theta}$ from collapsing to zero, so the filter never "locks in" and stops learning. It is a **forgetting factor** — a tunable adaptation rate.

- Large $\Sigma_{\tilde{r}}$ → fast adaptation, noisy/jittery estimates (tracks time-varying parameters). Small $\Sigma_{\tilde{r}}$ → slow, smooth convergence (best for truly constant parameters).

- **→ Intuition:** $r_k$ is the parameter filter's version of the tuning knob from [09](09_Making_the_KF_Bulletproof.ipynb) — it sets how quickly old data is discounted. Some schemes shrink $\Sigma_{\tilde r}$ over time to get fast early learning and stable late estimates.

### 🧩 Nonlinearity ⇒ Use an EKF or SPKF

- The output map $g(x_k,u_k,\theta_k)$ is almost always **nonlinear in $\theta$** (parameters multiply states, enter exponentials, etc.), so a linear KF won't do.

- Apply the **EKF** (Jacobian $\partial g/\partial\theta$) or, preferably, the **SPKF** (sigma points in parameter space — no derivatives) exactly as in [12](12_Extended_Kalman_Filter.ipynb)–[13](13_Sigma_Point_Unscented_Kalman_Filter.ipynb), with $\theta$ playing the role of the state.

  → The prediction step is trivial ($\hat\theta_k^- = \hat\theta_{k-1}^+$, $\Sigma_{\tilde\theta,k}^- = \Sigma_{\tilde\theta,k-1}^+ + \Sigma_{\tilde r}$ — random walk), so all the work is in the nonlinear **correction** that fits $\theta$ to the measured $d_k$.

- **→ Intuition:** parameter estimation is "system identification, recursively" — the filter tunes the model to the data online, updating with every new measurement instead of a batch fit.

### 🧩 Joint Estimation — One Big Filter

- **Stack** state and parameters into a single augmented state and run *one* filter:

$$
x_k^{\text{aug}} = \begin{bmatrix} x_k \\ \theta_k\end{bmatrix}, \qquad
x_{k+1}^{\text{aug}} = \begin{bmatrix} f(x_k,u_k,\theta_k,w_k) \\ \theta_k + r_k\end{bmatrix}, \qquad d_k = h(x_k,u_k,\theta_k,v_k).
$$

  → Estimate everything at once with a single (nonlinear) KF on the combined vector. The filter automatically tracks the **cross-covariance** between state and parameters.

- **+** Captures state–parameter correlations correctly; conceptually simple.
  **−** Larger state (cost grows with $L^3$); can be numerically delicate when state and parameters evolve on very different time scales.

- **→ Intuition:** "one filter to rule them all" — cleanest when state and parameters are tightly coupled and you can afford the bigger covariance matrix.

### 🧩 Dual Estimation — Two Coupled Filters

- Run **two filters in parallel**: a **state filter** (estimates $x_k$, treating current $\hat\theta$ as known) and a **parameter filter** (estimates $\theta_k$, treating current $\hat x$ as known). They exchange estimates each step.

$$
\text{State filter: } \hat{x}_k^+ = \text{KF}(\hat{x},\, \hat{\theta}_k^-), \qquad
\text{Param filter: } \hat{\theta}_k^+ = \text{KF}(\hat{\theta},\, \hat{x}_k^-).
$$

  → Two smaller filters instead of one big one, each feeding the other its latest estimate. Decouples the two problems for numerical stability and lower per-filter cost.

- **+** Smaller matrices, better conditioning across different time scales, modular.
  **−** Ignores (or only approximates) the state–parameter cross-covariance, so it can be slightly less statistically efficient than joint.

- **→ Intuition:** dual estimation is an **alternating optimization** — "fix parameters, update state; fix state, update parameters" — the recursive-filter analogue of coordinate descent. Often preferred in practice (e.g. battery/BMS identification) for its stability and modularity.

### 🧩 Summary

- Unknown model **parameters** are estimated by casting them as a **random-walk state** $\theta_{k+1}=\theta_k+r_k$ with an output $d_k=g(x_k,u_k,\theta_k)+e_k$.

- The artificial noise $r_k$ acts as a **forgetting factor**: it keeps the filter adapting and sets the learning rate.

- Because $g$ is nonlinear in $\theta$, use an **EKF or (preferably) SPKF**; the prediction step is a trivial random walk, all work is in the correction.

- **Joint** estimation augments $[x;\theta]$ into one filter (captures cross-covariance, larger/costlier); **dual** estimation runs coupled state and parameter filters (smaller, more stable, approximate coupling).

---
*Course 3 complete. Next: [15 · Bayesian Recursion, Monte-Carlo Integration & Importance Sampling](15_Bayesian_Recursion_MonteCarlo_Importance_Sampling.ipynb) — the road to particle filters.*